# Read the result and decide (chapter 12)

Read a pilot result against the signed lines, simulate how often the rule errs, then track the two monthly lines after purchase. The pilot gaps are synthetic.

In [ ]:
import numpy as np
import pandas as pd

KILL, PASS, SE_PLAN = 0.3, 0.53, 0.115
g = pd.read_csv("../data/pilot_pair_gaps.csv")
first = g[g["phase"] == "first_look"]
pair_fig = first.groupby("pair")["gap_points"].mean()
result = pair_fig.mean()
se = pair_fig.std() / np.sqrt(len(pair_fig))
print(pair_fig.round(2).to_string())
print(f"First look result {result:.2f} points; spread / sqrt(6) = {se:.3f} (plan {SE_PLAN})")

## The four readings (12.1)

In [ ]:
def read(x):
    return "Stop" if x < KILL else ("Pass, if the route to cash holds" if x >= PASS else "Extend once")

print("First look:", read(result))
all24 = g["gap_points"].mean()
print(f"After the extension all {len(g)} gaps average {all24:.2f}:", "pass" if all24 >= KILL else "stop")
for x in (0.12, 0.41, 0.56):
    print(f"{x}: {(x - KILL) / SE_PLAN:+.1f} noise units, {read(x)}")

## How often the rule errs

Independent pair gaps with spread 0.4 points; 12 gaps at the first look, 24 after the extension. A rough guide only.

In [ ]:
rng = np.random.default_rng(1203)

def simulate(true_gap, runs=200_000):
    gaps = rng.normal(true_gap, 0.4, size=(runs, 24))
    first = gaps[:, :12].mean(axis=1)
    final = gaps.mean(axis=1)
    stop_first = first < KILL
    pass_first = first >= PASS
    passed = pass_first | (~stop_first & ~pass_first & (final >= KILL))
    return stop_first.mean(), pass_first.mean(), passed.mean()

for true_gap in (0.48, 0.27, 0.15):
    s, p1, p = simulate(true_gap)
    print(f"True gap {true_gap}: wrong stop {s:.1%}, first look pass {p1:.1%}, ends in a pass {p:.1%}")

## Buy on the measured saving (12.3)

In [ ]:
hl = 100_000 * 0.0044
print(f"{hl:.0f} hL: {hl * 30:,.0f} at replacement cost, {hl * 110:,.0f} at the upper bound")

## The two monthly lines (12.4)

In [ ]:
month_hL = 8_330
recovered = round(month_hL * (0.024 - 0.020))
sold, off_plan, stock = 9, 18, 6
value = sold * 110 + off_plan * 30
running = 21_880 / 12
print(f"Line one: {recovered} hL recovered, {0.4 / 0.48:.0%} of the claim")
print(f"Line two: value {value:,}; share banked {(sold + off_plan) / recovered:.0%}; net {value - running:,.0f}")